# 03 · Text features

Two families, and the reasoning for each matters more than the code.

### 1 · Loughran-McDonald category rates

Counts of **Negative**, **Uncertainty**, **Litigious**, **Constraining** and
**Modal** words as a share of total words.

Loughran and McDonald (2011) built these lists because general-purpose
sentiment dictionaries misread financial text. In the Harvard-IV
psychosocial dictionary, *liability*, *tax*, *cost*, *capital*, *depreciation*
and *board* all score **negative** — in a 10-K they are ordinary accounting
vocabulary. Roughly three-quarters of Harvard-IV's negative hits in filings
are words that carry no negative meaning in a financial context.

Using a finance-specific dictionary is a defensible choice with a citable
reason. That is worth more in an interview than a more elaborate model you
cannot justify.

### 2 · Year-over-year risk-factor dissimilarity — the headline feature

Cosine **distance** between a firm's Item 1A this year and its own Item 1A
last year, on TF-IDF vectors.

Risk-factor sections are overwhelmingly boilerplate and are largely copied
forward year to year. A firm that rewrites a substantial share of it has
done something unusual, and the question this project asks is whether that
rewriting carries information about credit deterioration that the balance
sheet does not already contain.

Note it is **within-firm** by construction: it asks how much this company
changed relative to *itself*, so stable cross-firm differences in house
drafting style difference out. It is also strictly **backward-looking** —
year *t* is compared only to *t−1*, never to anything in the future.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import config, viz
viz.use_style()
print(config.describe())

## The dictionary

The full Loughran-McDonald master dictionary — 86,486 words with every
category — ships inside `pysentiment2`, so there is no manual download and
no dependency on a hosted file that might move.

In [ ]:
import importlib, text_features
importlib.reload(text_features)
from text_features import load_lm, LM_CATEGORIES, count_categories, USABLE

cats = load_lm()
print("statuses treated as usable:", USABLE)

Sanity-check the lists before trusting them. `shall` and `must` should be
strong-modal; `may` and `could` weak; `litigation` litigious; `impairment`
negative.

In [ ]:
probes = {"IMPAIRMENT":"Negative", "LITIGATION":"Litigious",
          "MAY":"Modal", "UNCERTAIN":"Uncertainty",
          "RESTRICTED":"Constraining", "PROFITABLE":"Positive"}
for w, expected in probes.items():
    hits = [c for c in LM_CATEGORIES if w in cats[c]]
    mark = "ok  " if expected in hits else "??  "
    print(f"{mark}{w:<12} expected {expected:<13} found {hits}")

## Compute both feature families

~2–4 minutes.

> **Kernel note.** `import x` is a no-op if `x` was already imported in
this kernel, so an edited `.py` file silently does not load and the cell
happily reports numbers from the previous version. Every cell below that
calls into a pipeline module reloads it first, in dependency order.

In [ ]:
import importlib
import sections, text_features
importlib.reload(sections)          # reload dependencies first
importlib.reload(text_features)

text_features.main()

## The headline feature's distribution

Expect a heavy mass near zero — most firms copy their risk factors forward
almost verbatim — with a right tail of genuine rewrites. **That shape is the
result**, not a defect: if `risk_change` were uniform, the feature would be
measuring drafting noise rather than events.

In [ ]:
df = pd.read_parquet(config.PROCESSED / "text_features.parquet")
rc = df["risk_change"].dropna()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(rc, bins=60, color=viz.SERIES[0], edgecolor=viz.SURFACE, linewidth=0.5)
ax.set_title("Year-over-year risk-factor dissimilarity")
ax.set_xlabel("1 − cosine similarity vs own prior year"); ax.set_ylabel("firm-years")
plt.show()

print(rc.describe(percentiles=[.01,.10,.25,.50,.75,.90,.99]).round(4).to_string())
print(f"\nobservations: {len(rc):,}")

### Does it move when the world moves?

A useful cross-check that needs no labels. 2020 filings describe a year in
which a pandemic arrived; if the feature is measuring anything real, 2020
should show more rewriting than neighbouring years. If it does not, the
feature is suspect.

In [ ]:
by_year = df.groupby("filing_year")["risk_change"].agg(["mean","median","count"]).dropna()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
yrs = by_year.index.to_numpy()
ax.plot(yrs, by_year["mean"], color=viz.SERIES[0], marker="o")
viz.label_end(ax, yrs[-1], by_year["mean"].iloc[-1], "  mean", viz.SERIES[0])
ax.plot(yrs, by_year["median"], color=viz.SERIES[1], marker="o")
viz.label_end(ax, yrs[-1], by_year["median"].iloc[-1], "  median", viz.SERIES[1])
ax.set_title("Risk-factor rewriting by filing year")
ax.set_xlabel("filing year"); ax.set_ylabel("1 − cosine similarity")
ax.set_xlim(yrs.min()-0.2, yrs.max()+0.6)
plt.show()

display(by_year.round(4))

### Read the tails by hand

The single most valuable check in this notebook. Pull the largest rewrites
and look at what those companies actually said. If the top of the
distribution is dominated by formatting changes or reorganised headings
rather than substantive new risks, the feature is measuring the wrong thing
and needs a normalisation step.

In [ ]:
sec = pd.read_parquet(config.INTERIM / "sections.parquet",
                      columns=["cik","filing_year","item1a","item1a_status"])
top = df.nlargest(5, "risk_change")[["cik","ticker","filing_year","risk_change"]]
display(top.drop(columns="cik"))

r = top.iloc[0]
def body(cik, yr):
    m = sec.query("cik == @cik and filing_year == @yr")
    return m.iloc[0]["item1a"] if len(m) else "(not found)"

print(f"=== {r.ticker} {r.filing_year}   risk_change = {r.risk_change:.3f} ===\n")
print("THIS YEAR :", body(r.cik, r.filing_year)[:300].replace("\n"," "), "\n")
print("PRIOR YEAR:", body(r.cik, r.filing_year-1)[:300].replace("\n"," "))

## Loughran-McDonald rates over time

Five series on one axis, all shares of total words, so they are directly
comparable. Direct labels rather than legend-only: three of these palette
slots sit below 3:1 contrast on this surface, and identity should never rest
on colour alone.

In [ ]:
cols = [f"lm_{c.lower()}" for c in ["Negative","Uncertainty","Litigious",
                                    "Constraining","Positive"]]
ts = df.groupby("filing_year")[cols].mean()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, c in enumerate(cols):
    ax.plot(ts.index, ts[c], color=viz.SERIES[i], marker="o", label=c[3:])
    viz.label_end(ax, ts.index[-1], ts[c].iloc[-1], "  " + c[3:], viz.SERIES[i])
ax.set_title("Loughran-McDonald category rates in Item 1A")
ax.set_xlabel("filing year"); ax.set_ylabel("share of words")
ax.set_xlim(ts.index.min()-0.2, ts.index.max()+1.4)
plt.show()

display(ts.round(5))

## Sanity gate

Everything below must hold before stage 4 uses these features.

In [ ]:
checks = {
  "risk_change within [0, 1]"          : bool(rc.between(0,1).all()),
  "first year always NaN"              : bool(df.loc[df.groupby("cik")["filing_year"]
                                              .transform("min").eq(df["filing_year"]),
                                              "risk_change"].isna().all()),
  "LM rates within [0, 1]"             : bool(df[cols].stack().between(0,1).all()),
  "no duplicate firm-years"            : bool(~df.duplicated(["cik","filing_year"]).any()),
  ">=2,000 year-over-year observations": bool(len(rc) >= 2000),
}
for k, v in checks.items():
    print(f"  {'ok  ' if v else 'FAIL'} {k}")
print("\n" + ("ALL PASS — ready for stage 4" if all(checks.values())
               else "FAILURES ABOVE — do not proceed"))